# Colab 実行環境チェック

GPU・メモリ・空き容量・PyTorch の状態を確認します。GPU がなくても実行できます。
上から順にセルを実行してください。実行結果には端末名やファイルパスが含まれることがあるので、共有前に確認してください。

In [ ]:
import importlib.util
import json
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone

def gib(n):
    return round(n / (1024 ** 3), 2)

report = {"checked_at_utc": datetime.now(timezone.utc).isoformat(),
          "python": sys.version.split()[0], "platform": platform.platform()}
disk = shutil.disk_usage('/content' if platform.system() == 'Linux' and __import__('os').path.isdir('/content') else '.')
report['disk_gib'] = {"total": gib(disk.total), "free": gib(disk.free)}
try:
    import psutil
    mem = psutil.virtual_memory()
    report['ram_gib'] = {"total": gib(mem.total), "available": gib(mem.available)}
except ImportError:
    report['ram_gib'] = 'psutil がないため取得できません'

print(json.dumps(report, ensure_ascii=False, indent=2))

## GPU と PyTorch
GPU ランタイムを有効化するには、Colab の **ランタイム → ランタイムのタイプを変更** を開きます。
割り当てられる GPU はセッションによって変わります。

In [ ]:
gpu = shutil.which('nvidia-smi')
if gpu:
    command = [gpu, '--query-gpu=name,memory.total,memory.free,driver_version', '--format=csv,noheader']
    result = subprocess.run(command, capture_output=True, text=True, timeout=15, check=False)
    report['nvidia_smi'] = result.stdout.strip() if result.returncode == 0 else result.stderr.strip()[:500]
else:
    report['nvidia_smi'] = '利用できません（CPU ランタイムでは正常です）'
print('nvidia-smi:', report['nvidia_smi'])

if importlib.util.find_spec('torch'):
    import torch
    report['torch'] = {"version": torch.__version__, "cuda_available": torch.cuda.is_available(),
                       "built_with_cuda": torch.version.cuda}
    if torch.cuda.is_available():
        devices = []
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            devices.append({"name": p.name, "vram_gib": gib(p.total_memory),
                            "capability": f'{p.major}.{p.minor}'})
        report['torch']['devices'] = devices
else:
    report['torch'] = 'インストールされていません'
print('PyTorch:', json.dumps(report['torch'], ensure_ascii=False, indent=2))

## 必要なライブラリを調べる
次のリストを自分のプロジェクトに合わせて編集してください。存在確認のみ行い、インストールはしません。

In [ ]:
modules = ['numpy', 'pandas', 'PIL', 'transformers', 'diffusers']  # 必要に応じて編集
report['modules'] = {}
for name in modules:
    try:
        present = importlib.util.find_spec(name) is not None
        report['modules'][name] = present
        print(f'{name}: {"利用可" if present else "未導入"}')
    except (ImportError, ValueError, ModuleNotFoundError) as exc:
        report['modules'][name] = False
        print(f'{name}: 確認できません ({exc})')

## レポートを保存（任意）
次のセルは Colab の一時領域に JSON を作ります。必要なら左のファイル欄からダウンロードしてください。

In [ ]:
from pathlib import Path
output_path = Path('colab_runtime_report.json')
output_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('保存先:', output_path.resolve())